# Diktor klon: ovoz + dumaloq gapiradigan video

Bu daftar ikki ish qiladi:
- **A. Ovoz** — uning darslaridagi ovozidan RVC modeli o‘qitiladi va tayyor diktor audiosi (Google AI Studio’da yasalgani) uning ovoziga aylantiriladi.
- **B. Yuz** — portret rasmi va shu ovozdan lab harakati mos keladigan video yasaladi (SadTalker).

**Boshlashdan oldin:** yuqorida `Runtime → Change runtime type → T4 GPU` ni tanlang.

Google Drive’da `Klon` papkasi bo‘lsin:
```
MyDrive/Klon/
   dataset/        ← uning dars audiolari (mp3/wav/m4a/mp4), jami 15–30 daqiqa
   diktor.mp3      ← Google AI Studio’da yasalgan diktor audiosi
   portret.png     ← uning rasmi (yuz to‘g‘ri, ko‘krakdan yuqori)
```
Natijalar `MyDrive/Klon/natija/` ga tushadi. Sessiya uzilib qolsa ham ish yo‘qolmaydi: qaytadan ishga tushirsangiz, qolgan joyidan davom etadi.

## 0. Google Drive’ni ulash

In [ ]:
#@title 0 · Drive’ni ulash va papkalarni tekshirish
from google.colab import drive
drive.mount('/content/drive')
import os, glob
K = '/content/drive/MyDrive/Klon'
os.makedirs(f'{K}/natija', exist_ok=True)
ds = [f for f in glob.glob(f'{K}/dataset/*') if os.path.isfile(f)]
print('dataset fayllari:', len(ds))
for f in ['diktor.mp3', 'portret.png']:
    print(f, '✓' if os.path.exists(f'{K}/{f}') else '✗ TOPILMADI')

## A. Ovoz klonlash (RVC · Applio)
A1 → A2 → A3 → A4 → A5 ketma-ket ishga tushiriladi. Eng uzog‘i A4 (o‘qitish), u 1–2 soat davom etadi.

In [ ]:
#@title A1 · Applio’ni o‘rnatish (≈5 daqiqa)
%cd /content
!git clone -q https://github.com/IAHispano/Applio /content/Applio
%cd /content/Applio
!apt -qq -y install portaudio19-dev > /dev/null
!pip -q install uv
!uv venv .venv --python 3.12 --seed -q
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --python /content/Applio/.venv/bin/python
!/content/Applio/.venv/bin/python core.py prerequisites --models --pretraineds-hifigan --no-exe
# o‘qitish natijalari Drive’da saqlansin (sessiya uzilsa ham yo‘qolmaydi)
import os
os.makedirs('/content/drive/MyDrive/Klon/rvc_logs', exist_ok=True)
if os.path.isdir('/content/Applio/logs') and not os.path.islink('/content/Applio/logs'):
    !cp -rn /content/Applio/logs/* /content/drive/MyDrive/Klon/rvc_logs/ 2>/dev/null
    !rm -rf /content/Applio/logs
if not os.path.exists('/content/Applio/logs'):
    os.symlink('/content/drive/MyDrive/Klon/rvc_logs', '/content/Applio/logs')
print('✓ tayyor')

In [ ]:
#@title A2 · Ovoz namunalarini tayyorlash (mono 44.1 kHz WAV, jami 30 daqiqagacha)
import glob, os, subprocess, json
K = '/content/drive/MyDrive/Klon'
OUT = '/content/dataset_wav'; os.makedirs(OUT, exist_ok=True)
LIMIT = 30 * 60   # soniya
total = 0
for i, f in enumerate(sorted(glob.glob(f'{K}/dataset/*'))):
    if total >= LIMIT: break
    pr = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'json', f], capture_output=True, text=True)
    dur = float(json.loads(pr.stdout or '{}').get('format', {}).get('duration', 0) or 0)
    if dur <= 0: print('o‘qilmadi:', f); continue
    take = min(dur, LIMIT - total)
    # past chastotali g‘uvillashni kesish + ovoz balandligini tenglash
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', f, '-t', str(take), '-vn', '-ac', '1', '-ar', '44100',
                    '-af', 'highpass=f=70,loudnorm=I=-18:TP=-2', f'{OUT}/{i:03d}.wav'])
    total += take
    print(f'{os.path.basename(f)}: {take/60:.1f} daq')
print(f'JAMI: {total/60:.1f} daqiqa', '✓' if total >= 600 else '⚠ 10 daqiqadan kam — sifat pastroq bo‘lishi mumkin')

In [ ]:
#@title A3 · Namunalarni bo‘laklash va xususiyatlarni ajratish (≈5–10 daqiqa)
MODEL = 'diktor'  #@param {type:"string"}
%cd /content/Applio
!.venv/bin/python core.py preprocess --model-name {MODEL} --dataset-path /content/dataset_wav --sample-rate 40000 --cpu-cores 2 --cut-preprocess Automatic --chunk-len 3.0 --overlap-len 0.3 --normalization-mode post
!.venv/bin/python core.py extract --model-name {MODEL} --f0-method rmvpe --sample-rate 40000 --cpu-cores 2 --gpu 0 --embedder-model contentvec
!.venv/bin/python core.py index --model-name {MODEL} --index-algorithm Auto
print('✓ tayyor')

In [ ]:
#@title A4 · Modelni o‘qitish (1–2 soat). Uzilsa — A1, keyin shu katakni qayta ishga tushiring, davom etadi
MODEL = 'diktor'  #@param {type:"string"}
EPOCH = 200  #@param {type:"integer"}
%cd /content/Applio
!.venv/bin/python core.py train --model-name {MODEL} --vocoder HiFi-GAN --save-every-epoch 10 --save-only-latest --total-epoch {EPOCH} --sample-rate 40000 --batch-size 8 --gpu 0 --pretrained --index-algorithm Auto
!ls -la /content/Applio/logs/{MODEL}/*.pth /content/Applio/logs/{MODEL}/*.index

In [ ]:
#@title A5 · Diktor audiosini uning ovoziga aylantirish
#@markdown **PITCH**: diktor audiosi **ayol** ovozida bo‘lsa `-12`, **erkak** ovozida bo‘lsa `0`.
#@markdown Avval **SINOV = True** bilan 40 soniyani tinglang, yoqsa **False** qilib to‘liq audioni aylantiring.
MODEL = 'diktor'  #@param {type:"string"}
PITCH = 0  #@param {type:"integer"}
SINOV = True  #@param {type:"boolean"}
import glob, os, subprocess
from IPython.display import Audio, display
K = '/content/drive/MyDrive/Klon'
L = f'/content/Applio/logs/{MODEL}'
pths = sorted([p for p in glob.glob(f'{L}/{MODEL}*.pth')], key=os.path.getmtime)
idx = [p for p in glob.glob(f'{L}/*.index') if 'trained' not in os.path.basename(p)] or glob.glob(f'{L}/*.index')
assert pths and idx, 'Model topilmadi — A4 tugaganini tekshiring'
print('model:', os.path.basename(pths[-1]), '| index:', os.path.basename(idx[0]))
src = '/content/diktor_in.wav'
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', f'{K}/diktor.mp3'] + (['-ss', '60', '-t', '40'] if SINOV else []) + ['-ac', '1', '-ar', '44100', src])
out = f'{K}/natija/' + ('sinov_klon.wav' if SINOV else 'diktor_klon.wav')
%cd /content/Applio
!.venv/bin/python core.py infer --input-path {src} --output-path "{out}" --pth-path "{pths[-1]}" --index-path "{idx[0]}" --pitch {PITCH} --f0-method rmvpe --index-rate 0.75 --protect 0.33 --volume-envelope 1.0 --export-format WAV
display(Audio(out))
print('✓ saqlandi:', out)

## B. Dumaloq gapiradigan video (SadTalker)
A5 da **to‘liq** audio aylantirilgandan keyin ishga tushiring. Audio 2 daqiqalik bo‘laklarga bo‘linib ishlanadi, har bo‘lak tayyor bo‘lishi bilan Drive’ga saqlanadi. Sessiya uzilsa B1 ni, keyin B3 ni qayta ishga tushiring: tayyor bo‘laklar o‘tkazib yuboriladi.

In [ ]:
#@title B1 · SadTalker’ni o‘rnatish (≈6 daqiqa)
%cd /content
!pip -q install uv
!git clone -q https://github.com/OpenTalker/SadTalker /content/SadTalker
%cd /content/SadTalker
!uv venv /content/st_venv --python 3.10 --seed -q
!uv pip install -q --python /content/st_venv/bin/python torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!uv pip install -q --python /content/st_venv/bin/python setuptools wheel numpy==1.23.5
!uv pip install -q --python /content/st_venv/bin/python --no-build-isolation -r requirements.txt
!bash scripts/download_models.sh > /dev/null 2>&1; ls checkpoints gfpgan/weights

In [ ]:
#@title B2 · Portretni tayyorlash (kvadrat 512×512)
from PIL import Image, ImageOps
K = '/content/drive/MyDrive/Klon'
im = ImageOps.exif_transpose(Image.open(f'{K}/portret.png')).convert('RGB')
w, h = im.size; s = min(w, h)
# yuqori qismini saqlab kvadratga kesish (bosh va yelka)
im = im.crop(((w - s) // 2, 0, (w - s) // 2 + s, s)).resize((512, 512), Image.LANCZOS)
im.save('/content/portret512.png'); display(im.resize((256, 256)))

In [ ]:
#@title B3 · Gapiradigan videoni yasash (24 daqiqalik audio ≈ 1–2 soat)
import glob, os, subprocess, shutil
K = '/content/drive/MyDrive/Klon'
CH = f'{K}/natija/yuz_bolaklar'; os.makedirs(CH, exist_ok=True)
AUD = '/content/aud_chunks'; os.makedirs(AUD, exist_ok=True)
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', f'{K}/natija/diktor_klon.wav', '-ac', '1', '-ar', '16000',
                '-f', 'segment', '-segment_time', '120', f'{AUD}/c%03d.wav'])
chunks = sorted(glob.glob(f'{AUD}/c*.wav'))
print('bo‘laklar:', len(chunks))
%cd /content/SadTalker
for c in chunks:
    name = os.path.basename(c)[:-4]
    dst = f'{CH}/{name}.mp4'
    if os.path.exists(dst): print(name, '— tayyor, o‘tkazildi'); continue
    tmp = f'/content/st_out/{name}'; shutil.rmtree(tmp, ignore_errors=True)
    !/content/st_venv/bin/python inference.py --driven_audio {c} --source_image /content/portret512.png --result_dir {tmp} --still --preprocess full --size 256 --batch_size 4 --expression_scale 1.0
    mp4 = sorted(glob.glob(f'{tmp}/**/*.mp4', recursive=True), key=os.path.getmtime)
    assert mp4, f'{name}: video chiqmadi — xatoni yuqoridan ko‘ring'
    shutil.copy(mp4[-1], dst); print(name, '✓')
# bo‘laklarni bitta videoga birlashtirish
with open('/content/list.txt', 'w') as f:
    for c in chunks: f.write(f"file '{CH}/{os.path.basename(c)[:-4]}.mp4'\n")
!ffmpeg -y -loglevel error -f concat -safe 0 -i /content/list.txt -c:v libx264 -crf 18 -pix_fmt yuv420p -an -r 25 "{K}/natija/yuz_video.mp4"
print('✓ tayyor:', f'{K}/natija/yuz_video.mp4')

## Tayyor bo‘lgach
`MyDrive/Klon/natija/` papkasidan ikkita faylni yuboring:
- `diktor_klon.wav` — uning ovozidagi diktor audiosi
- `yuz_video.mp4` — gapiradigan yuz videosi

Qolganini (dumaloq oyna, oltin ramka, video bilan birlashtirish) men qilaman.